# 🌍 TripForge: AI-Powered Multi-Agent Travel Planner
### Interactive Demonstration & Code Showcase Notebook

This notebook provides an interactive walkthrough and code showcase of **TripForge**, an autonomous multi-agent travel concierge system built for the Kaggle *Vibecoding Agents Capstone Project*.

TripForge combines Google's **Agent Development Kit (ADK)** with a custom **Model Context Protocol (MCP)** server to generate hyper-personalized, day-by-day travel itineraries. The system monitors live weather, travel logistics, and active city disruptions (like strikes or closures) and replans itineraries dynamically.

---

## 🏆 Key Concepts Demonstrations

| Rubric Concept | Location in Codebase / Notebook | Description |
| :--- | :--- | :--- |
| **Agent / Multi-agent system (ADK)** | `tripforge/agents/` / Section 4 | Four specialized agents (`ProfileAgent`, `ResearchAgent`, `ItineraryAgent`, `DisruptionAgent`) orchestrated sequentially. |
| **MCP Server** | `tripforge/mcp_server/` / Section 3 | Python Model Context Protocol server exposing travel tools over stdio. |
| **Security features** | `tripforge/utils/security.py` / Section 2 | Symmetric Fernet profile encryption, HMAC-SHA256 call signatures, PII scrubbing, and consent guards. |
| **Deployability** | `app.py` / Section 5 | Read-only filesystem workarounds (logs/downloads fallbacks) for Vercel/serverless environments. |
| **Agent skills** | `tripforge/orchestrator.py` / Section 4.3 | Interactive generator streaming progress events and rendering plans. |


## ⚙️ 1. Setup & Environments
First, let's configure the system path so we can import the `tripforge` modules directly and verify our environment is ready.

In [ ]:
import os
import sys
import json
import asyncio
from datetime import datetime

# Ensure project root is in the Python search path
project_root = os.path.abspath(".")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"[*] Project root directory: {project_root}")
print(f"[*] Python Executable: {sys.executable}")

# Check requirements are satisfied
try:
    import google.adk
    import mcp
    import cryptography
    import flask
    import rich
    print("[+] Core dependencies imported successfully!")
except ImportError as e:
    print(f"[-] Missing dependency: {e}. Please run 'pip install -r requirements.txt'")

## 🔒 2. Implemented Security Features
TripForge integrates strict security and privacy controls to satisfy capstone requirements. Let's showcase how they work:

### 2.1 Hardware-derived Symmetric Encryption
We encrypt traveler profiles using `cryptography.fernet` using a stable key derived from the local machine fingerprint (CPU, Username, Hostname) without saving keys to disk.

In [ ]:
from tripforge.utils.security import encrypt_profile, decrypt_profile_content, get_machine_key

mock_profile = {
    "destination": "Paris",
    "days": 3,
    "travelers": 2,
    "budget": 2000.0,
    "currency": "USD",
    "accessibility_needs": "None",
    "dietary_restrictions": "Gluten-Free"
}

# 1. Derive machine key
machine_key = get_machine_key()
print(f"[*] Derived Machine Key (Base64): {machine_key.decode()[:20]}...")

# 2. Encrypt Profile
encrypted_data = encrypt_profile(mock_profile)
print(f"[*] Encrypted Profile Data (Bytes): {encrypted_data[:40]}...")

# 3. Decrypt Profile
decrypted_profile = decrypt_profile_content(encrypted_data)
print(f"[+] Successfully Decrypted Profile:")
print(json.dumps(decrypted_profile, indent=2))

### 2.2 PII Log Scrubbing
To ensure traveler privacy, logs are automatically scrubbed of sensitive PII (emails, phone numbers, passport-like ID digits) before writing or display.

In [ ]:
from tripforge.utils.security import scrub_pii

raw_log_entry = (
    "Booking flight for John Doe (Passport: US987654321, Phone: +1-555-0199-823) "
    "and confirmation sent to john.doe@example.com."
)

scrubbed_log = scrub_pii(raw_log_entry)
print("--- Raw Log Entry ---")
print(raw_log_entry)
print("\n--- Scrubbed Log Entry ---")
print(scrubbed_log)

### 2.3 HMAC-SHA256 Tool Call Verification
To prevent malicious agents or prompt injection attacks from executing unauthorized local tools on the MCP server, every tool request is signed by the orchestrator with an HMAC signature containing parameters and verified by the server.

In [ ]:
from tripforge.utils.security import sign_tool_call, verify_tool_call

tool_name = "get_weather"
tool_params = {"city": "Tokyo", "date": "2026-07-15"}

# Sign tool call
signature = sign_tool_call(tool_name, tool_params)
print(f"[*] Generated HMAC Signature: {signature}")

# Verify valid signature
is_valid = verify_tool_call(tool_name, tool_params, signature)
print(f"[+] Verification check with original parameters: {is_valid}")

# Try to tamper with parameters
tampered_params = {"city": "Tokyo", "date": "2026-07-15", "extra_injected_cmd": "rm -rf /"}
is_tampered_valid = verify_tool_call(tool_name, tampered_params, signature)
print(f"[-] Verification check with tampered parameters: {is_tampered_valid} (Should be False)")

## 🔌 3. Model Context Protocol (MCP) Server
TripForge exposes travel tools to the agents via a Model Context Protocol (MCP) server. Let's inspect the tools exposed in the FastMCP server located in [travel_tools_server.py](file:///d:/codingProject/GITHUB/Kaggle/tripforge/mcp_server/travel_tools_server.py).

The MCP server exposes:
1. `get_weather(city, date)`: Fetches weather forecasts or uses seasonal fallbacks.
2. `search_activities(city, accessibility_required, dietary_preference, category)`: Searches local activities database.
3. `get_country_info(country_name)`: Returns currencies, language, visa requirements, tipping customs, and emergency numbers.
4. `estimate_transport_cost(origin_city, destination_city, transport_type)`: Compares costs, duration, carbon footprint (CO2) of flights/trains.
5. `check_disruption(city, activity_name, date)`: Detects active strikes, weather alerts, or closures.

Let's call the tool handlers directly from the notebook to demonstrate their outputs:

In [ ]:
# Import the tools directly from travel_tools_server to see their responses
from tripforge.mcp_server.travel_tools_server import get_weather, get_country_info, check_disruption

# Call weather tool
weather_res = await get_weather(city="Paris", date="2026-08-15")
print("--- Weather Tool Response ---")
print(json.dumps(weather_res, indent=2))

# Call country info tool
country_res = await get_country_info(country_name="France")
print("\n--- Country Info Tool Response ---")
print(json.dumps(country_res, indent=2))

# Call disruption checker tool
disruption_res = await check_disruption(city="Paris", activity_name="Louvre Museum", date="2026-06-27")
print("\n--- Disruption Check Tool Response ---")
print(json.dumps(disruption_res, indent=2))

## 🤖 4. Multi-Agent System (Google ADK)
TripForge implements a multi-agent hierarchy using Google's **Agent Development Kit (ADK)**. It contains 4 specialized agents working in sequence:

1. **ProfileAgent**: Validates raw user input parameters, infers travel styles, and flags critical accessibility/dietary safety constraints.
2. **ResearchAgent**: Queries MCP tools to gather climate forecasts, activity catalogs, transit options, and country-level guides.
3. **ItineraryAgent**: Designs a day-by-day itinerary balancing budget, energy constraints, diet, weather, and checks for disruptions.
4. **DisruptionAgent**: Monitors active disruptions and dynamically replans affected segments using alternative activities.

Let's run a live simulation of this multi-agent orchestrator in **Mock Mode** (using mock LLM responses to enable instant offline execution).
We will print each stream event live as the agents collaborate!

In [ ]:
from tripforge.orchestrator import stream_tripforge

async def run_live_planning_simulation():
    print("[*] Launching multi-agent pipeline stream...")
    
    # We will plan a trip using the stream generator
    # We specify TRIPFORGE_MODE=mock to run offline without requiring active Gemini API credentials
    os.environ["TRIPFORGE_MODE"] = "mock"
    
    generator = stream_tripforge(
        destination="Paris",
        days=3,
        travelers=2,
        budget=2000.0,
        currency="USD",
        interests=["culture", "food", "history"]
    )
    
    async for event in generator:
        event_type = event.get("type")
        if event_type == "progress":
            step = event.get("step")
            msg = event.get("message")
            icon = event.get("icon", "🤖")
            print(f"\n{icon} [Step {step}/4] {msg}")
        elif event_type == "agent_start":
            print(f"    -> Agent {event.get('agent')} started reasoning...")
        elif event_type == "agent_stream":
            # Print agent's reasoning snippets
            content = event.get("content", "")
            print(f"\033[92m{content}\033[0m", end="", flush=True)
        elif event_type == "complete":
            print("\n\n[+] Multi-Agent Planning Completed Successfully!")
            print(f"Summary Weather Conditions: {event.get('summary', {}).get('weather_condition')}")
            print(f"Total Calculated Spend: {event.get('summary', {}).get('currency')} {event.get('summary', {}).get('total_cost')}")
            
            # Print first 25 lines of the generated itinerary
            itinerary = event.get("itinerary", "")
            print("\n--- Generated Itinerary Snippet ---")
            print("\n".join(itinerary.splitlines()[:25]))
            print("...")

# Run the async simulator
await run_live_planning_simulation()

## 🌐 5. Deployability & Production Considerations

### 5.1 Read-Only Filesystem Adaptation
Serverless endpoints (such as Vercel) restrict write access except for the `/tmp` directory. 
TripForge is designed to run in serverless contexts:
- **Logger**: Catches `OSError` with `errno == 30` (Read-only filesystem) when attempting to write `tripforge.log`, and silently redirects outputs to `sys.stderr` which cloud platforms automatically capture.
- **Downloads**: If local folder creation fails (like `/scratch`), downloads automatically fall back to `tempfile.gettempdir()`.

Let's test the logging error handler with a mock permission error:

In [ ]:
import errno

def log_error_simulation(error: Exception):
    """Simulates the robust Vercel read-only log error handler."""
    try:
        # Simulate trying to write to a read-only directory
        raise OSError(errno.EROFS, "Read-only file system")
    except OSError as e:
        if e.errno == 30: # errno.EROFS
            print("[+] Caught Read-Only Filesystem error (errno 30) - Silently ignored, stdout/stderr logs preferred.")
        else:
            print(f"[-] Unhandled OS error: {e}")
    except Exception as e:
        print(f"[-] Other error: {e}")

log_error_simulation(OSError(30, "Read-only file system"))

### 5.2 Python 3.12 Regex Boundaries
In Python 3.12, double dollar signs `$$` inside search patterns are evaluated as zero-width matches at *every* character location, leading to infinite loops or empty matches in non-greedy patterns. 
TripForge uses single `$` anchors and boundary conditions to guarantee stable execution on modern Python environments:

In [ ]:
import re

# Safe regex pattern using single $ for string end anchors
pattern_safe = re.compile(r"---PROFILE_JSON---(.*)---SUMMARY---$", re.DOTALL)
sample_text = "---PROFILE_JSON---\n{\n  \"destination\": \"Paris\"\n}\n---SUMMARY---"

match = pattern_safe.search(sample_text)
if match:
    print("[+] Successfully parsed sections using Python 3.12 compliant boundary pattern!")
    print(f"Extracted Profile: {match.group(1).strip()}")

## 🏁 Conclusion
This notebook demonstrates:
1. **Security**: Strong profile cryptography, hardware-derived keys, PII scrubbers, and signed tool interfaces.
2. **MCP integration**: Standalone MCP subprocess connecting agents to tools.
3. **Multi-Agent Flow**: Sequentially orchestrated Google ADK agents with rich logging and mock fallbacks.
4. **Deployability**: Vercel/serverless read-only file systems protection and Python 3.12 compatibility.